# Visualisation and analysis data 

#### hbp-d000017_PatchClamp-GranuleCells_pub

In [1]:
import os
import re
import glob
import zipfile
import numpy as np
from neo.io import get_io
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from neo import Epoch

### Find_path | Find_zip_file | unzip_file

In [2]:
def find_path(working_directory, dataset_name): # Find the folder where the zip file is 
    """
    input: working directory: str: path #os.path.join(os.getcwd())
            dataset_name: str: name of the dataset EBRAINS same as the folder name
    output: path of the folder
    """
    dataset_name = dataset_name.replace(" ", "_")
    search_pattern = os.path.join(working_directory, f"{dataset_name}")
    dataset_directory = glob.glob(search_pattern)

    if dataset_directory:
        return dataset_directory[0] # type(files) -> list
    else:
        return f"No folder or file '{dataset_name}' in '{working_directory}'."
    
def find_zip_file(dataset_directory, dataset_name): # Find zip file in the folder
    zip_path = glob.glob(os.path.join(dataset_directory, '*.zip'))
    return zip_path[0]

def unzip_file(zip_path, dataset_directory): # Unzip and extract the zipfile in the chosen directory
    # Unzip folder (run one time) !!!
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(dataset_directory)


### Order data 

In [ ]:
df_path_subject_id = pd.DataFrame(columns=('subject_id', 'path_data', 'files')) # create the dataframe
df_path_subject_id['subject_id'] = [folder for folder in list_folders if folder.startswith('GrC') and os.path.isdir(os.path.join(dataset_directory, folder))] # get the id of each Subject
df_path_subject_id['path_data'] = df_path_subject_id['subject_id'].apply(lambda x: os.path.join(dataset_directory, x)) # get the path of the folder of each subject
df_path_subject_id['files'] = df_path_subject_id['path_data'].apply(os.listdir) # get in a list the data files 
df_path_subject_id.head()

dict_data = {}
for index, row in df_path_subject_id.iterrows():
    folder_path = row['path_data']
    file_names = row['files']
    
    # Loop through each file name in the 'name' list
    for file_name in file_names:
        file_path = f"{folder_path}/{file_name}"
        # Read the file into a DataFrame
        file_df = get_io(file_path)
        data_neo = file_df.read()
        dict_data[file_name] = data_neo

In [ ]:
# Get Protocol CC step 

dict_data_CC = {}
for index, row in df_path_subject_id.iterrows():
    folder_path = row['path_data']
    file_names = row['files']

    for file in file_names:
        if 'CC step' in file:
            file_path = f"{folder_path}/{file}"
            file_df = get_io(file_path)
            data_neo = file_df.read()
            dict_data_CC[file] = data_neo

# Get Protocol IV - 70. 
dict_data_IV_70= {}

for index, row in df_path_subject_id.iterrows():
    folder_path = row['path_data']
    file_names = row['files']

    for file in file_names:
        if 'IV -70' in file:
            file_path = f"{folder_path}/{file}"
            file_df = get_io(file_path)
            data_neo = file_df.read()

            dict_data_IV_70[file] = data_neo


# Get Protocol EPSC. 
dict_data_EPSP = {}

for index, row in df_path_subject_id.iterrows():
    folder_path = row['path_data']
    file_names = row['files']

    for file in file_names:
        if 'EPSP' in file:
            file_path = f"{folder_path}/{file}"
            file_df = get_io(file_path)
            data_neo = file_df.read()
            dict_data_EPSP[file] = data_neo

### Plot data | plot data zoom 

In [ ]:

def plot_data(file_path):
    data = get_io(file_path).read(lazy=True)
    for segment in data[0].segments:
        signal = segment.analogsignals[0].load()
        plt.plot(signal.times, signal)
    #plt.xlim(0.0, 1) # MODIFIED REGULARLY
    #plt.ylim(-72,-60)
    plt.xlabel(f"Time ({signal.times.units.dimensionality})")
    plt.ylabel(f"Membrane potential (signal.units.dimensionality)")

def plot_data_zoom(file_path):
    data = get_io(file_path).read(lazy=True)
    for segment in data[0].segments:
        signal = segment.analogsignals[0].load()
        plt.plot(signal.times - signal.t_start, signal)
        plt.xlim(0.05, 0.5)
        plt.xlabel(f"Time ({signal.times.units.dimensionality})")
        plt.ylabel(f"Membrane potential ({signal.units.dimensionality})")


In [4]:
# PLot the onset all the file for one protocol

for file in dict_data_CC.keys():
    signal = dict_data_CC[file][0].segments[0].analogsignals[0]

    plt.title('CC step')
    plt.xlim(0.05,0.15)

    plt.legend(dict_data_CC.keys(),bbox_to_anchor=(1.55, 1.0),loc='upper right')
    plt.plot(signal.times, signal) 


#for file in dict_data_CC.keys():
for segment in dict_data_CC['240216-B_0004 CC step.abf'][0].segments:
    signal = segment.analogsignals[0]
    signalv = np.array(signal)
    plt.plot(signal.times, signalv)
    #plt.xlim(60.09,60.135)
    plt.xlim(0.05,0.15)
    plt.ylim(-80, -10)
    plt.axvline(x=0.09085, color='r')

NameError: name 'dict_data_CC' is not defined

### Finding onset 

In [ ]:
#Création d'une fonction qui permet d'avoir le début de la dépolarisation et la fin de l'hyperpolarisation
def onset(file_path):
    #Création de 3 liste vide
    var_signal0 = []
    var_signal1 = []
    var_signal = []
    data = get_io(file_path).read(lazy=True)
    for segment in data[0].segments:
        signal = segment.analogsignals[0].load()
        signalv = np.array(signal)
        #moyenne du repos
        mean = sum(signalv[:10])/len(signalv[:10])
        print('mean')
        #intervalle des valeurs considérer au repos
        interp = mean + 3
        interm = mean - 3
        #récupération des indices ou le signal change
        onset_test = signalv
        var_signal= [i for i, x in enumerate(signalv) if x > interp or x < interm]
        print(var_signal)
        #si il y a une valeur dans var_signal alors on rajoute la 1er valeur
        #dans var_signal0 = début de la variation du signal
        #la dernière valeur dans var_signal1 = fin de la variation du signal
        if len(var_signal) > 0:
            var_signal0.append(var_signal[0])
            var_signal1.append(var_signal[-1])
            #on récupère la plus petit valeur dans chaque signal
    var0 = min(var_signal0)
    var1 = min(var_signal1)
    return signal.times[var0]-signal.times[0], signal.times[var1]-signal.times[0]



### Finding sampling frequency 

In [3]:
# Get sampling rate For all protocols

print('-------')
print('signal_CC')
for file in dict_data_CC.keys():
    signal_CC = dict_data_CC[file][0].segments[0].analogsignals[0].sampling_rate
    print(signal_CC)

print('-------')
print('signal_IV_70')

for file in dict_data_IV_70.keys():
    signal_IV_70 = dict_data_IV_70[file][0].segments[0].analogsignals[0].sampling_rate
    print(signal_IV_70)

print('-------')
print('signal_EPSP')
for file in dict_data_EPSP.keys():
    signal_EPSP = dict_data_EPSP[file][0].segments[0].analogsignals[0].sampling_rate
    print(signal_EPSP)


-------
signal_CC


NameError: name 'dict_data_CC' is not defined